# Lab 05: Xây dựng mô hình Multinomial Naive Bayes

## Mục tiêu
- Hiểu cơ chế hoạt động của thuật toán **Multinomial Naive Bayes**.
- Cài đặt mô hình Multinomial Naive Bayes từ đầu bằng Python, không dùng thư viện học máy.
- Thực hiện dự đoán và đánh giá hiệu quả mô hình.
- So sánh với mô hình `MultinomialNB` của sklearn.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Cài đặt lớp Multinomial Naive Bayes
- Cài đặt lớp `MultinomialNaiveBayes` với 02 phương thức chính:
    + `__init__(self, alpha=1.0)`: Khởi tạo mô hình với tham số làm mịn Laplace `alpha`.
    + `fit(X, y)`: Huấn luyện mô hình trên dữ liệu huấn luyện.
    + `predict(X)`: Dự đoán nhãn cho dữ liệu đầu vào.
- Cài đặt các phương thức phụ trợ cần thiết để tính toán xác suất tiên nghiệm và xác suất có điều kiện với Laplace smoothing.
    +  `compute_class_prior(self, y)`: Tính xác suất tiên nghiệm P(y).
    +  `compute_likelihood(self, X, y)`: Tính xác suất có điều kiện P(x|y) với Laplace smoothing.
    +  `compute_posterior(self, X, y)`: Tính xác suất hậu nghiệm P(y|x) với định lý Bayes.


In [2]:
class MultinomialNaiveBayes:
    # Khởi tạo mô hình với tham số alpha cho Laplace smoothing
    def __init__(self, alpha=1):
        self.class_priors = {}      # Xác suất tiên nghiệm P(y)
        self.likelihoods = {}       # Hàm khả năng P(X|y)
        self.class_counts = {}      # Số lượng mẫu của mỗi lớp
        self.n_samples = 0          # Số lượng mẫu huấn luyện
        self.n_features = 0         # Số lượng đặc trưng
        self.vocab_size = 0         # Kích thước từ điển
        self.alpha = alpha          # Hệ số Laplace smoothing
        self.classes = None

    # Hàm tính xác suất tiên nghiệm P(y)
    def compute_class_prior(self, y):
        classes, counts = np.unique(y, return_counts=True)
        self.classes = classes
        self.n_samples = len(y)
        for cls, count in zip(classes, counts):
            self.class_counts[cls] = count
            self.class_priors[cls] = count / self.n_samples

    # Hàm khả năng likelihood P(X|y) với Laplace smoothing (alpha=1)
    def compute_likelihood(self, X, y):
        self.n_features = X.shape[1]
        self.vocab_size = self.n_features

        # Khởi tạo ma trận đếm từ cho từng lớp
        for cls in self.classes:
            X_cls = X[y == cls]
            word_counts = X_cls.sum(axis=0)  # Tổng số lần xuất hiện của từng từ trong lớp đó
            total_count = word_counts.sum()

            # Tính xác suất có điều kiện P(w_i | c)
            likelihood = (word_counts + self.alpha) / (total_count + self.alpha * self.vocab_size)
            self.likelihoods[cls] = likelihood

    # Hàm tính xác suất hậu nghiệm P(y|X) ∝ P(y) * Π P(x_i|y)
    def compute_posterior(self, X):
        posteriors = []
        for x in X:
            class_probs = {}
            for cls in self.classes:
                log_prior = np.log(self.class_priors[cls])
                log_likelihood = np.sum(x * np.log(self.likelihoods[cls]))
                log_posterior = log_prior + log_likelihood
                class_probs[cls] = log_posterior
            posteriors.append(class_probs)
        return posteriors

    # Huấn luyện mô hình
    def fit(self, X, y):
        self.compute_class_prior(y)
        self.compute_likelihood(X, y)

    # Dự đoán nhãn cho dữ liệu đầu vào
    def predict(self, X):
        posteriors = self.compute_posterior(X)
        y_pred = [max(posterior, key=posterior.get) for posterior in posteriors]
        return np.array(y_pred)


# 2. Kiểm tra mô hình trên các tập dữ liệu

## 2.1. Tập dữ liệu nb_samples.csv

In [3]:
df = pd.read_csv('nb_samples.csv')
df

,message,label
0,Dear Friend Lunch,N
1,Dear Friend,N
2,Dear Lunch,N
3,Dear Friend Lunch,N
4,Dear Friend,N
5,Dear Friend,N
6,Dear,N
7,Dear Money,N
8,Dear Money,S
9,Dear Money,S


## 2.1.Tập dữ liệu SMS Spam Collection Dataset
- Tải tập dữ liệu SMS Spam Collection Dataset từ Kaggle (https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset).
- Chọn lọc cột chứa gán nhãn (v1) là `spam` (thư rác) hoặc `ham` (thư bình thường) và cột chứa nội dung tin nhắn (v2).
- Chuyển nhãn `spam` thành 1 và `ham` thành 0.
- Tiền xử lý dữ liệu văn bản *(sử dụng thư viện `re` của Python)*:
    + Chuyển tất cả các tin nhắn về chữ thường.
    + Loại bỏ các ký tự đặc biệt, số và dấu câu.
    + Loại bỏ khoảng trắng thừa.
    + Tách từ (tokenization) nếu cần thiết.
    + Loại bỏ stop words nếu cần thiết.
- Tạo ma trận đặc trưng sử dụng Bag of Words với `CountVectorizer` của sklearn *(đã bao gồm các bước tiền xử lý văn bản)*.

In [4]:
# Đọc dữ liệu
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [5]:
# Tiền xử lý văn bản
import re

def preprocess_text(text):
    # Chuyển về chữ thường
    text = text.lower()
    # Loại bỏ ký tự đặc biệt, số và dấu câu
    text = re.sub(r'[^a-z\s]', '', text)
    # Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    # Tách từ (tokenization) nếu cần thiết
    text = text.split()
    # Loại bỏ stop words nếu cần thiết
    stop_words = set(['the', 'is', 'in', 'and', 'to', 'a'])
    text = [word for word in text if word not in stop_words]
    return text

df['tokens'] = df['message'].apply(preprocess_text)
df.head()

,label,message,tokens
0,0,"Go until jurong point, crazy.. Available only ...","[go, until, jurong, point, crazy, available, o..."
1,0,Ok lar... Joking wif u oni...,"[ok, lar, joking, wif, u, oni]"
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, wkly, comp, win, fa, cup, final,..."
3,0,U dun say so early hor... U c already then say...,"[u, dun, say, so, early, hor, u, c, already, t..."
4,0,"Nah I don't think he goes to usf, he lives aro...","[nah, i, dont, think, he, goes, usf, he, lives..."


In [6]:
# Tạo ma trận đặc trưng
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['message']).toarray()
y = df['label'].values
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [7]:
# Phân chia dữ liệu thành tập huấn luyện và tập kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Tập huấn luyện:", X_train.shape)
print("Tập kiểm tra:", X_test.shape)

Tập huấn luyện: (4457, 8404)
Tập kiểm tra: (1115, 8404)


In [8]:
# Tạo và huấn luyện mô hình Naive Bayes
custom_model = MultinomialNaiveBayes(alpha=1.0)
model = custom_model.fit(X_train, y_train)

# Dự đoán trên tập kiểm tra
print("Class Prior:",custom_model.class_priors)
print("Likelihood:",custom_model.likelihoods)
y_pred_custom = custom_model.predict(X_test)

# Đánh giá mô hình
accuracy_custom = accuracy_score(y_test, y_pred_custom)
report_custom = classification_report(y_test, y_pred_custom)
matrix_custom = confusion_matrix(y_test, y_pred_custom)
print("Accuracy:", accuracy_custom)
print("Classification Report:", report_custom)
print("Confusion Matrix:\n", matrix_custom)

Class Prior: {0: 0.8660533991474085, 1: 0.13394660085259144}
Likelihood: {0: array([2.82645562e-05, 2.82645562e-05, 5.65291125e-05, ...,
       2.82645562e-05, 2.82645562e-04, 5.65291125e-05]), 1: array([5.02652890e-04, 1.34040771e-03, 5.58503211e-05, ...,
       1.11700642e-04, 5.58503211e-05, 5.58503211e-05])}
Accuracy: 0.9802690582959641
Classification Report:               precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.93      0.93      0.93       150

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
 [[954  11]
 [ 11 139]]


# 3. So sánh với MultinominalNB của sklearn
- Sử dụng `MultinomialNB` từ sklearn để huấn luyện và dự đoán trên cùng tập dữ liệu.
- So sánh kết quả với mô hình tự cài đặt

In [9]:
# Tạo và huấn luyện mô hình Naive Bayes với MultinomialNB của sklearn
sklearn_model = MultinomialNB()
sklearn_model.fit(X_train, y_train)

# Dự đoán trên tập kiểm tra
y_pred_sklearn = sklearn_model.predict(X_test)

# Đánh giá mô hình
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)
report_sklearn = classification_report(y_test, y_pred_sklearn)
matrix_sklearn = confusion_matrix(y_test, y_pred_sklearn)
print("Accuracy:", accuracy_sklearn)
print("Classification Report:", report_sklearn)
print("Confusion Matrix:\n", matrix_sklearn)

Accuracy: 0.9802690582959641
Classification Report:               precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.93      0.93      0.93       150

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
 [[954  11]
 [ 11 139]]
